In [1]:
# SPDX-FileCopyrightText: Copyright 2025 Idiap Research Institute <contact@idiap.ch>
# SPDX-FileContributor: Anshul Gupta <anshul.gupta@idiap.ch>
# SPDX-License-Identifier: GPL-3.0-only

"""
Post-process gaze following prediction for LAEO
"""
import numpy as np
import torch
import pickle
import itertools
import io
from sklearn.metrics import precision_score, recall_score, f1_score

In [2]:
class CPU_Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == "torch.storage" and name == "_load_from_bytes":
            return lambda b: torch.load(io.BytesIO(b), map_location="cpu")
        else:
            return super().find_class(module, name)


def load_pkl(path_to_file):
    with open(path_to_file, "rb") as file:
        return CPU_Unpickler(file).load()


def to_numpy(tens):
    try:
        return tens.cpu().numpy()
    except:
        return tens.numpy()


def squeeze_res(res):
    new_res = {}
    for k, v in res.items():
        try:
            new_res[k] = v.squeeze(0)
        except AttributeError:
            new_res[k] = v
    return new_res


def is_inside(head_bbox, gaze_pt):
    if (
        gaze_pt[0] > head_bbox[0]
        and gaze_pt[0] < head_bbox[2]
        and gaze_pt[1] > head_bbox[1]
        and gaze_pt[1] < head_bbox[3]
    ):
        return True
    return False

In [3]:
# Process one scene
def process_laeo(res):
    res = squeeze_res(res)
    head_boxes = to_numpy(res["head_bboxes"])
    num_people = head_boxes.shape[0]
    gaze_preds = to_numpy(res["gp_pred"])

    laeo_gts = to_numpy(res["laeo_gt"]).tolist()
    laeo_gts_ = []
    laeo_preds = []

    # get laeo pred
    pairs = np.array(list(itertools.permutations(torch.arange(num_people), 2)))
    for pair_id, pair in enumerate(pairs):  # iterate over pairs
        if laeo_gts[pair_id] == -1:
            continue

        head_bbox1 = head_boxes[pair[0]].tolist()
        gaze_pred1 = gaze_preds[pair[0]].tolist()

        head_bbox2 = head_boxes[pair[1]].tolist()
        gaze_pred2 = gaze_preds[pair[1]].tolist()

        if is_inside(head_bbox1, gaze_pred2) and is_inside(head_bbox2, gaze_pred1):
            laeo_pred = 1
        else:
            laeo_pred = 0

        laeo_preds.append(laeo_pred)
        laeo_gts_.append(laeo_gts[pair_id])
    return laeo_gts_, laeo_preds

In [4]:
# Prcoess LAEO per dataset
def compute_p_r_laeo_dataset(test_preds, dataset=None):
    if dataset == None:
        dataset_test_preds = test_preds
    else:
        dataset_test_preds = [dt for dt in test_preds if dt["dataset"][0] == dataset]
    print(f"Info: Len Preds {dataset}: {len(dataset_test_preds)}")

    all_laeo_gts = []
    all_laeo_preds = []
    for res in dataset_test_preds:
        # Process Laeo
        laeo_gts_, laeo_preds = process_laeo(res)
        # Store
        all_laeo_gts.extend(laeo_gts_)
        all_laeo_preds.extend(laeo_preds)

    print(f"Info: Len Combined Preds {dataset}: {len(all_laeo_preds)}")

    if sum(all_laeo_gts) == 0 or sum(all_laeo_preds) == 0:
        print(
            "Warning: Precision and Recall are ill-defined and being set to 0.0 due to no predicted samples ..."
        )
        return 0.0, 0.0

    # Calculate precision and recall
    precision = precision_score(all_laeo_gts, all_laeo_preds)
    recall = recall_score(all_laeo_gts, all_laeo_preds)
    f1 = f1_score(all_laeo_gts, all_laeo_preds)

    return precision, recall, f1

In [5]:
# Process LAEO for all the datatses
path_to_results = ".../test_predictions.p"
test_preds = load_pkl(path_to_results)

/tmp/ipykernel_3924533/1310678334.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return lambda b: torch.load(io.BytesIO(b), map_location="cpu")


In [6]:
datasets = [None]
for dataset in datasets:
    precision, recall, f1 = compute_p_r_laeo_dataset(test_preds, dataset)
    print(f"Precision on {dataset}: {precision:.3f}")
    print(f"Recall on {dataset}: {recall:.3f}")
    print(f"F1 on {dataset}: {f1:.4f}")
    print("-" * 16)

Info: Len Preds None: 392
Info: Len Combined Preds None: 920
Precision on None: 0.968
Recall on None: 0.852
F1 on None: 0.9064
----------------
